PREPARING THE WORKSPACE

In [ ]:
import os
import xml.etree.ElementTree as ET
from lxml import etree
import time
import google.generativeai as genai  # Call the Gemini API
from tqdm.notebook import tqdm  # For the progress bar

In [ ]:
# --- SETTINGS ---

# Gemini Configuration
with open('gemini.txt', 'r') as f:  # Remember to place your API key in the gemini.txt file
    API_KEY = f.read()
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",  # Select your preferred model, you can see the available ones here: https://ai.google.dev/gemini-api/docs/models
    system_instruction=(
        "You are a Senior Data Engineer. Your task is to document SSIS flows in PLAIN TEXT. "
        "CRITICAL RULE: The order of the XML is not the order of execution. You must analyze "
        "'DTS:PrecedenceConstraint' to reconstruct the logical flow step by step. "
        "Ignore missing or disabled components. Use the provided template."
    )
)

# Path: Remember to change the folder paths
# where your ssis packages will be along with your parameters file and your conmgr files
# and the path where you want the documentation files to be loaded.
ssis_location = "[PATH_TO_YOUR_DTSX_PARAMS_AND_CONMGRS]"
temp_output = "[TEMPORARY_PATH_TO_SAVE_SUMMARIES_OF_YOUR_PARAM_AND_CONMGRS]"
docutentation_output_location = "[PATH_WHERE_DOCUMENTATION_OF_DTSX_WILL_BE_SAVED]"

ruta_docu_conmgr = temp_output+"conmgr_summary.txt"
ruta_docu_params = temp_output+"params_summary.txt"

PREPARING CLASSES AND FUNCTIONS

In [ ]:
# CLASS WITH FUNCTIONS TO PROCESS THE Project.params FILE AND THE .conmgr

class SSISParser:
    def __init__(self, project_path):
        self.project_path = project_path
        self.ns = {'DTS': 'www.microsoft.com/SqlServer/Dts'}
        self.namespaces = {
            'SSIS': 'www.microsoft.com/SqlServer/SSIS',
            'DTS': 'www.microsoft.com/SqlServer/Dts'
        }

    def parse_conmgr(self):
        output = []
        output.append("-" * 60)
        output.append("--- CONNECTIONS (.conmgr) ---")
        output.append("-" * 60)
        
        for file in os.listdir(self.project_path):
            if file.endswith(".conmgr"):
                try:
                    tree = ET.parse(os.path.join(self.project_path, file))
                    root = tree.getroot()

                    name = root.get(f"{{{self.ns['DTS']}}}ObjectName")
                    
                    expr_node = root.find(".//DTS:PropertyExpression[@DTS:Name='ConnectionString']", self.ns)
                    expression = expr_node.text.strip() if expr_node is not None else "No dynamic expression"

                    obj_data_conn = root.find(".//DTS:ObjectData/DTS:ConnectionManager", self.ns)
                    static_conn = "Not found"
                    if obj_data_conn is not None:
                        static_conn = obj_data_conn.get(f"{{{self.ns['DTS']}}}ConnectionString")

                    output.append(f"File: {file}")
                    output.append(f"Object Name: {name}")
                    output.append(f"Expression (Logic): {expression}")
                    output.append(f"Static Value: {static_conn}")
                    output.append("-" * 30)
                
                except Exception as e:
                    output.append(f"Error processing {file}: {e}")
        
        return "\n".join(output)

    def parse_params(self):
        output = []
        output.append("--- PROJECT PARAMETERS ---")
        param_file = os.path.join(self.project_path, "Project.params")
        
        if os.path.exists(param_file):
            tree = ET.parse(param_file)
            root = tree.getroot()
            for param in root.findall(".//SSIS:Parameter", self.namespaces):
                name = param.get("{www.microsoft.com/SqlServer/SSIS}Name")
                value_node = param.find("./SSIS:Properties/SSIS:Property[@SSIS:Name='Value']", self.namespaces)
                value = value_node.text if value_node is not None else "NULL/Empty"
                output.append(f"Parameter: {name} | Initial value: {value}")
        else:
            output.append("Project.params file not found.")
            
        return "\n".join(output)

parser = SSISParser(ssis_location)

In [ ]:
# FUNCTION THAT CLEANS THE DTSX XML TO REDUCE SIZE AND "NOISE"
def clean_dtsx(input_path):
    ns = {'DTS': 'www.microsoft.com/SqlServer/Dts'}
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(input_path, parser)
    root = tree.getroot()

    # 1. Remove Visual Metadata (90% of the weight)
    for node in root.xpath("//DTS:DesignTimeProperties", namespaces=ns):
        node.getparent().remove(node)

    # 2. Remove Disabled Tasks
    # We look for the Disabled="True" attribute at any level
    for node in root.xpath("//*[@DTS:Disabled='True']", namespaces=ns):
        node.getparent().remove(node)

    return etree.tostring(tree, encoding='unicode', pretty_print=False)

In [ ]:
# FUNCTION TO DOCUMENT SSIS PACKAGES (DTSX)
def document_all_packages(folder_path, params_ref, conmgr_ref, l_packages):
    # Load the text summaries (the ones you already have)
    with open(params_ref, 'r', encoding="utf-8") as f: p_text = f.read()
    with open(conmgr_ref, 'r', encoding="utf-8") as f: c_text = f.read()

    # Prompt to be sent to the AI
    prompt = f"""
    Document the attached txt file, which contains cleaned code from a dtsx type file.
    
    GLOBAL CONTEXT:
    PARAMETERS: {p_text}
    CONNECTIONS: {c_text}
    
    INSTRUCTION: Rebuild the execution order using precedence constraints.
    If a task has no input, it is the start. Follow the arrows (Constraints) to the end.

    Format Instructions:
    1. Do NOT include greetings, introductions or filler text ("Here is your analysis...").
    2. Do NOT use code blocks (```markdown, ```text, etc). Return plain text directly.
    3. Follow strictly the provided template below.
    4. If a field does not apply, write "N/A".

    Analysis Instructions:
    - 1. PROCESS SUMMARY: Describe what it does based on the logic.
    - 2. INFRASTRUCTURE AND CONNECTIVITY (ACTIVE): Identify only connections used by enabled tasks.
    - 3. RELATED PROJECT PARAMETERS: List the parameters that the package uses, indicate the name, data type and briefly explain its use. If you detect that it is not used in the package, mark it as "(NOT USED)".
    - 4. CONTROL FLOW (ONLY ENABLED TASKS): Describe each active task in the Control Flow. MAKE SURE to only document tasks that are ENABLED/ACTIVE.
    - 5. DATA FLOW (DATA FLOW LOGIC): Describe each Data Flow Task. MAKE SURE to only document flows that are ENABLED/ACTIVE.
    
    Use the following template exactly:

    --- TEMPLATE START ---
    TECHNICAL REPORT OF ETL PROCESS PACKAGE NAME: [Name]
    
    1. PROCESS SUMMARY:

    
    2. INFRASTRUCTURE AND CONNECTIVITY (ACTIVE)
    a) Connection 1: [Name]
    -Type: [SQL Server / Excel / etc]
    -Server/Path: [Name or path]
    -Database: [Resolved value]
    
    b) Connection 2: [Name]
    -Type: [SQL Server / Excel / etc]
    -Server/Path: [Name or path]
    -Database: [Resolved value]

    c) ...

    
    3. RELATED PROJECT PARAMETERS
    - Parameter1: Explanation...
    - Parameter2: Explanation... (NOT USED)

    
    4. CONTROL FLOW (ONLY ENABLED TASKS):    
    a) TASK: [Name]
    - TYPE: [Execute SQL Task / Data Flow / etc]
    - LOGIC: [If it is SQL, put the code here. If it is another, describe action]
    - PRECEDENCE: [Which task or data flow was executed previously, if there is no precedence put "N/A"]

    b) TASK: [Name]
    - TYPE: [Execute SQL Task / Data Flow / etc]            
    - LOGIC: [If it is SQL, put the code here. If it is another, describe action]
    - PRECEDENCE: [Which task or data flow was executed previously]

    c)...

    
    5. DATA FLOW (DATA FLOW LOGIC):
    - SOURCE: [Table or Query]            
    - TRANSFORMATIONS: [Mappings, Derived Columns, active Lookups. Consider the order of steps]
    - FILTERS: Indicate the most used filters throughout the process, that is what is placed in the 'where' of the select. These usually represent business rules.
    - DESTINATION: [Final table and database]
    - PRECEDENCE: [Which task or data flow was executed previously]

    """

    for file in tqdm(l_packages):
        path = os.path.join(folder_path, file)
        
        # Clean XML
        clean_xml = clean_dtsx(path)
        temp_clean_path = f"{docutentation_output_location}//temp_{file}.txt"
        with open(temp_clean_path, "w", encoding="utf-8") as f:
            f.write(clean_xml)

        # Upload to the File API
        sample_file = genai.upload_file(path=temp_clean_path)
        while sample_file.state.name == "PROCESSING":
            time.sleep(2)
            sample_file = genai.get_file(sample_file.name)

        try:
            response = model.generate_content([sample_file, prompt])            
            # Save result
            output_name = docutentation_output_location+"//DOCU__"+file.replace(".dtsx", ".txt")
            with open(output_name, "w", encoding="utf-8") as f:
                f.write(response.text)
        
        except Exception as e:
            print(f"Critical error in package {file}: {e}")
            # continue # Skip to the next package
        
        finally:  # This is executed ALWAYS, whether there is an error or not
            # Cleanup
            if os.path.exists(temp_clean_path):
                os.remove(temp_clean_path)
            try:
                genai.delete_file(sample_file.name)
            except:
                pass

PROCESS THE FILES

In [ ]:
# SELECT THE DTSX FILES THAT WILL BE DOCUMENTED
my_packages = [f for f in os.listdir(ssis_location) if f.endswith('.dtsx')]
# my_packages = my_packages[:2] # Uncomment this line if you only want to process a sample of packages (useful for testing)
print("#packages: "+str(len(my_packages)))
my_packages

In [ ]:
# PROCESS AND SUMMARIZE THE .conmgr FILES
conmgr_docu = parser.parse_conmgr()
with open(ruta_docu_conmgr, "w", encoding="utf-8") as new_archivo:
    new_archivo.write(conmgr_docu)
print(conmgr_docu)

In [ ]:
# PROCESS AND SUMMARIZE THE Project.params FILE TO LATER SAVE IT IN A TXT
params_docu = parser.parse_params()
with open(ruta_docu_params, "w", encoding="utf-8") as new_archivo:
    new_archivo.write(params_docu)
print(params_docu) 

In [ ]:
# DOCUMENT
document_all_packages(ssis_location, ruta_docu_params, ruta_docu_conmgr, my_packages)